# ⚡ Gated Recurrent Unit (GRU) Network
**Efficient Time Series Forecasting with Keras/TensorFlow**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow version : {tf.__version__}')
print('Libraries loaded ✅')

## 2. Load & Explore Dataset
> Using a **Synthetic Time Series** dataset with trend, seasonality, and noise. This complexity allows us to fairly compare GRU performance against LSTM.

In [ ]:
# Generate dataset
np.random.seed(42)
t = np.linspace(0, 50, 1000)
# Trend + Seasonality + Noise
y = 0.05 * t + 5 * np.sin(0.2 * t) + 2 * np.cos(0.5 * t) + np.random.randn(1000) * 0.5
df = pd.DataFrame({'time': t, 'value': y})

print(f'Shape   : {df.shape}')
df.head()

## 3. Time Series Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['time'], df['value'], color='#f59e0b', lw=1.5, label='Value')
ax.set_title('Synthetic Time Series (Trend + Seasonality + Noise)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time'); ax.set_ylabel('Value')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Data Preprocessing
> GRUs require 3D input: `(batch_size, sequence_length, features)`. We create sequences using a sliding window. Scaling is critical for RNN/GRU convergence.

In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 30
values = df['value'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_values = scaler.fit_transform(values)

train_size = int(len(scaled_values) * 0.8)
train_data = scaled_values[:train_size]
test_data = scaled_values[train_size - SEQ_LENGTH:]

X_train, y_train = create_sequences(train_data, SEQ_LENGTH)
X_test, y_test = create_sequences(test_data, SEQ_LENGTH)

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape : {X_test.shape}')

## 5. What makes GRU special?
> The **Gated Recurrent Unit (GRU)** is a streamlined variant of the LSTM. It addresses the vanishing gradient problem while being computationally more efficient:

1. **Update Gate**: Combines the LSTM's forget and input gates. It decides how much past information to keep and how much new information to add.
2. **Reset Gate**: Decides how much past information to forget when computing the new candidate memory.
3. **Single State**: Merges the cell state and hidden state into a single hidden state, reducing parameters by ~25% compared to LSTM.

## 6. Build GRU Model

In [ ]:
def build_gru(units=64, dropout=0.2, seq_length=30, layers=1):
    model = keras.Sequential(name='GRU_Model')
    
    for i in range(layers):
        return_seq = True if i < layers - 1 else False
        input_shape = (seq_length, 1) if i == 0 else None
        model.add(layers.GRU(units, return_sequences=return_seq, input_shape=input_shape))
        if i < layers - 1:
            model.add(layers.Dropout(dropout))
            
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

model = build_gru(units=64, dropout=0.2, seq_length=SEQ_LENGTH, layers=1)
model.summary()

## 7. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, verbose=1)
]

start_time = time.time()
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)
print(f'Training completed in {time.time() - start_time:.2f} seconds')

## 8. Training History

In [ ]:
hist = pd.DataFrame(history.history)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist['loss'], color='#e05252', lw=2, label='Train')
ax.plot(hist['val_loss'], color='#f59e0b', lw=2, label='Val')
ax.set_title('Training Loss (MSE)', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Best Val Loss: {hist["val_loss"].min():.4f}')

## 9. Evaluate on Test Set

In [ ]:
y_pred = model.predict(X_test).flatten()
y_pred_inv = scaler.inverse_transform(y_pred.reshape(-1, 1))
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

mse = mean_squared_error(y_test_inv, y_pred_inv)
mae = mean_absolute_error(y_test_inv, y_pred_inv)

print('='*50)
print('          GRU Test Set Results')
print('='*50)
print(f'  MSE : {mse:.4f}')
print(f'  MAE : {mae:.4f}')
print('='*50)

## 10. Actual vs Predicted Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
test_indices = np.arange(len(y_test_inv))
ax.plot(test_indices, y_test_inv, color='#52a8e0', lw=1.5, label='Actual')
ax.plot(test_indices, y_pred_inv, color='#e05252', lw=1.5, linestyle='--', label='Predicted')
ax.set_title('Test Set: Actual vs Predicted (GRU)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Step'); ax.set_ylabel('Value')
ax.legend()
plt.tight_layout(); plt.show()

## 11. Multi-Step Forecasting

In [ ]:
forecast_steps = 50
last_seq = scaled_values[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)
forecast_scaled = []

# Autoregressive forecasting
for _ in range(forecast_steps):
    pred = model.predict(last_seq, verbose=0)
    forecast_scaled.append(pred[0, 0])
    last_seq = np.append(last_seq[:, 1:, :], np.reshape(pred, (1, 1, 1)), axis=1)

forecast_inv = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1))
last_actual_inv = scaler.inverse_transform(scaled_values[-SEQ_LENGTH:])

fig, ax = plt.subplots(figsize=(14, 5))
hist_steps = 100
ax.plot(np.arange(hist_steps), last_actual_inv[-hist_steps:], color='#52a8e0', lw=2, label='Historical')
ax.plot(np.arange(hist_steps-1, hist_steps-1+forecast_steps), 
        np.vstack((last_actual_inv[-1], forecast_inv)), 
        color='#e05252', lw=2, linestyle='--', label='Forecast')
ax.axvline(hist_steps-1, color='gray', linestyle=':', alpha=0.5, label='Forecast Start')
ax.set_title(f'{forecast_steps}-Step Ahead Forecast (GRU)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Step'); ax.set_ylabel('Value')
ax.legend()
plt.tight_layout(); plt.show()

## 12. GRU vs LSTM: Performance & Speed Comparison

In [ ]:
# Compare GRU and LSTM on the same data
results = {}

for model_type in ['GRU', 'LSTM']:
    print(f'Training {model_type}...')
    if model_type == 'GRU':
        m = build_gru(units=64, dropout=0.2, seq_length=SEQ_LENGTH, layers=1)
    else:
        m = build_gru(units=64, dropout=0.2, seq_length=SEQ_LENGTH, layers=1) # Reuse function but we'll manually swap layer type for fairness
        # Actually, let's build LSTM explicitly
        m = keras.Sequential([
            layers.LSTM(64, input_shape=(SEQ_LENGTH, 1)),
            layers.Dropout(0.2),
            layers.Dense(1)
        ])
        m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    
    cb = [EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]
    
    start = time.time()
    m.fit(X_train, y_train, validation_split=0.15, epochs=80, batch_size=32, callbacks=cb, verbose=0)
    train_time = time.time() - start
    
    yp = m.predict(X_test).flatten()
    yp_inv = scaler.inverse_transform(yp.reshape(-1, 1))
    mse = mean_squared_error(y_test_inv, yp_inv)
    params = m.count_params()
    
    results[model_type] = {'MSE': mse, 'Time (s)': train_time, 'Params': params}
    print(f'  -> MSE: {mse:.4f}, Time: {train_time:.2f}s, Params: {params}')

res_df = pd.DataFrame(results).T
print('\nComparison Summary:')
display(res_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(res_df.index, res_df['MSE'], color=['#f59e0b', '#34d399'], edgecolor='white')
axes[0].set_title('Test MSE (Lower is Better)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Squared Error')

axes[1].bar(res_df.index, res_df['Time (s)'], color=['#f59e0b', '#34d399'], edgecolor='white')
axes[1].set_title('Training Time (Lower is Better)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Seconds')
plt.tight_layout(); plt.show()

## 13. Save Model & Scaler

In [ ]:
import os, joblib
os.makedirs('../models', exist_ok=True)
model.save('../models/gru_model.keras')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model saved  → models/gru_model.keras')
print('Scaler saved → models/scaler.pkl')

## 14. Key Takeaways
> - **GRUs are highly efficient**: They often match LSTM performance with ~25% fewer parameters and faster training/inference times.
> - **Simplified gating**: The merge of the cell/hidden state and forget/input gates makes GRUs easier to tune and less prone to overfitting on smaller datasets.
> - **When to choose GRU**: Default to GRU for most sequence tasks. Reserve LSTM for highly complex, very long sequences where the dedicated cell state provides a distinct advantage.
> - **Autoregressive forecasting** accumulates error over time; the further you forecast, the wider the confidence interval should be.